In [ ]:
import sys  # sys: 현재 Python 환경 정보를 담은 내장 모듈
!{sys.executable} -m pip install tensorflow
# sys.executable: 지금 노트북이 실제로 사용하는 Python 경로
# -m pip install: 그 Python의 pip으로 tensorflow 설치
# !{}: 셸 명령어를 실행할 때 Python 변수를 중괄호로 넣는 문법

## 역전파
- pseudo code


- 순전파
```python
for 배치 in 전체데이터:
   x, y = 배치    # 입력


   z1 = x @ w1 + b1    # 첫번째 층 통과
   a1 = activation(z1)  # 첫번째 층 활성화 함수 적용


   z2 = a1 @ w2 + b2
   y_pred = activation(z2)  # 출력층 활성화 함수 통과


   loss = loss_fn(y, y_pred)  # 실제정답, 예측값의 차이를 계산
```


-역전파
'''python
for 배치 in 전체데이터:
    y_hat = 순전파(x)
    loss = 손실함수(y, y_hat) #오차계산
    gradients = 역전파(loss) # 역전파로 기울기를 계산
    W = W - learning_rate * gradients # 가중치를 업데이트
'''


## 기울기 소실

In [ ]:
import numpy as np  # numpy: 수학 계산 라이브러리, np라는 별명으로 불러옴

def sigmoid(z):  # sigmoid 함수 정의: z를 받아 0~1 사이 확률로 변환
    return 1 / (1 + np.exp(-z))  # 공식: 1 / (1 + e^(-z))

# 시그모이드 함수의 도함수(미분값) 정의
def sigmoid_deriv(z):  # 도함수: 기울기가 얼마나 되는지 계산
    s = sigmoid(z)  # 먼저 sigmoid 값을 구함
    return s * (1 - s)  # 도함수 공식: σ(z) × (1 - σ(z)), z=0이면 0.5×0.5=0.25

z_vals = np.linspace(-5, 5, 1000)  # -5~5 사이를 1000등분한 배열 생성
max_deriv = sigmoid_deriv(z_vals).max()  # 1000개 도함수값 중 최댓값 추출
print(max_deriv)  # 출력: 0.25 (sigmoid 도함수의 이론적 최대값)

# 시그모이드 함수에서 기울기의 최댓값은 0.25
# 3층짜리 신경망에서 활성화 함수로 시그모이드를 사용하면:
#   1층 기울기 최대: 0.25
#   2층 기울기 최대: 0.25 × 0.25 = 0.0625  ← 절반 이하로 줄어듦
#   3층 기울기 최대: 0.25 × 0.25 × 0.25 = 0.015625  ← 거의 0


In [ ]:
0.25 * 0.25 * 0.25  # 3층 신경망에서 기울기 최대치: 0.015625 (역전파 시 곱해짐)

In [ ]:
# 층이 더해질수록 기울기가 소실된다.
# 역전파에서 기울기를 업데이트하려 할 때, 기울기가 소실되면
# 너무 작아서 앞쪽 층의 가중치 업데이트가 거의 이루어지지 않는다.
# 기울기 소실을 막기 위해 ReLU가 등장한다.

In [ ]:
# ReLU (Rectified Linear Unit): 음수는 0, 양수는 그대로
def relu(z):  # z: 입력값
    return max(0, z)  # 0과 z 중 큰 값 반환 → 음수면 0, 양수면 그대로

test_vals = [-5, -3, 0, 3, 5]  # 테스트할 입력값 목록
for z in test_vals:  # 각 값에 relu 적용
    print(f"{z}: {relu(z)}")  # f-string: 변수를 {} 안에 넣어 출력

# 양수 구간(z>0)에서 도함수=1 → 기울기가 1 그대로 전달됨 (소멸 없음)
# 음수 구간(z<0)에서 도함수=0 → Dead ReLU 문제 (LeakyReLU로 해결)
# Keras 사용법: Dense(100, activation='relu')

패션 MNIST

In [ ]:
import tensorflow as tf  # tensorflow: 딥러닝 프레임워크, tf라는 별명 사용
import keras              # keras: tf 안에 포함된 고수준 API (층·모델 설계)

# 버전 확인: 두 버전이 호환되는지 확인 (불일치 시 오류 발생)
print("TF  :", tf.__version__)    # TF 버전 출력 (예: 2.21.0)
print("Keras:", keras.__version__)  # Keras 버전 출력 (예: 3.14.1)

In [ ]:
(train_input, train_target), (test_input, test_target) = keras.datasets.fashion_mnist.load_data()
# keras.datasets.fashion_mnist: 옷 사진 데이터셋 (10종류, 28×28 흑백)
# load_data(): 데이터를 (훈련용, 테스트용) 두 쌍으로 반환
# train_input: 훈련 사진 60,000장, shape=(60000, 28, 28)
# train_target: 훈련 정답 레이블, shape=(60000,), 값 범위 0~9
# test_input/test_target: 테스트용 10,000장 (지금은 안 씀)

In [ ]:
import sys  # 현재 Python 경로를 얻기 위한 내장 모듈
!{sys.executable} -m pip install matplotlib  # 그래프 그리는 라이브러리 설치

In [ ]:
import matplotlib.pyplot as plt  # matplotlib: 그래프·이미지 시각화 라이브러리

fig, axs = plt.subplots(1, 10, figsize=(10, 10))
# plt.subplots(행, 열): 1행 10열의 subplot(작은 그림 틀) 생성
# figsize=(10, 10): 전체 그림 크기 (가로 10인치, 세로 10인치)
# fig: 전체 캔버스, axs: 10개 틀의 배열

for i in range(10):  # i = 0, 1, 2, ..., 9 (10번 반복)
    axs[i].imshow(train_input[i], cmap='gray')
    # axs[i]: i번째 틀
    # imshow(): 이미지 표시 함수
    # train_input[i]: i번째 사진 (shape: 28×28)
    # cmap='gray': 컬러맵을 흑백으로 지정 (없으면 초록빛으로 나옴)
    axs[i].axis('off')  # 축(숫자 눈금) 숨기기 - 사진만 깔끔하게 보이도록

plt.tight_layout()  # subplot 간격을 자동으로 맞춰 겹치지 않게 조정
plt.show()          # 화면에 그림 출력

In [ ]:
train_target[:10]  # 처음 10개 정답 레이블 확인
# 숫자 0~9가 의류 종류를 나타냄
# 0=티셔츠, 1=바지, 2=스웨터, 3=드레스, 4=코트
# 5=샌들, 6=셔츠, 7=스니커즈, 8=가방, 9=부츠

In [ ]:
train_scaled = train_input / 255.0
# 픽셀값 0~255를 0~1 사이로 정규화
# 255.0으로 나누는 이유: 픽셀 최대값이 255이므로 나누면 0~1 범위가 됨
# 정규화 목적: 큰 숫자는 학습을 불안정하게 만들기 때문

train_scaled = train_scaled.reshape(-1, 28*28)
# reshape: 배열 형태를 바꿈 (데이터 값은 그대로, 구조만 변경)
# -1: '행 수는 알아서 계산해줘' (60000장이면 60000)
# 28*28 = 784: 28×28 이미지를 784칸 한 줄로 펼침
# 결과 shape: (60000, 784)

train_scaled.shape  # 변환 결과 확인: (60000, 784) 출력 예상

In [ ]:
keras.layers.Input(shape=(784,))
# Input 층: 모델에 들어올 데이터의 형태를 선언
# shape=(784,): 한 샘플이 784개의 숫자로 이루어짐
# 뒤의 콤마(,): 784짜리 1차원 튜플임을 파이썬에 알려주는 문법
# 이 층은 연산 없음 - 입력 형태 지정만 담당

In [ ]:
input_layer = keras.layers.Input(shape=(784,))  # 입력층: 784칸 벡터
output_layer = keras.layers.Dense(units=10, activation='softmax')
# Dense: 완전연결층 (이전 층 모든 뉴런과 연결)
# units=10: 출력 뉴런 10개 (옷 종류 10가지)
# activation='softmax': 10개 출력을 합이 1인 확률로 변환

model = keras.Sequential([input_layer, output_layer])
# Sequential: 층을 순서대로 쌓는 그릇
# [input_layer, output_layer]: 입력층 → 출력층 순서로 쌓음

In [ ]:
model.compile(
    optimizer='adam',
    # optimizer: 가중치를 어떻게 업데이트할지 결정하는 공부법
    # 'adam': 학습률을 자동 조절하는 옵티마이저 (기본값으로 잘 작동)

    loss='sparse_categorical_crossentropy',
    # loss: 예측이 얼마나 틀렸는지 재는 채점 기준
    # 'sparse_categorical_crossentropy': 정답이 정수(0~9)일 때 사용
    # (정답이 [0,0,1,...] 형태 원핫이면 'categorical_crossentropy' 사용)

    metrics=['accuracy']
    # metrics: 학습 중 눈으로 볼 평가 지표
    # 'accuracy': 정답률 (0~1 사이, 높을수록 좋음)
)

In [ ]:
model.fit(
    train_scaled,   # 훈련 입력 데이터 (shape: 48000, 784)
    train_target,   # 훈련 정답 레이블 (shape: 48000,)
    epochs=5        # 전체 데이터를 5번 반복해서 학습
                    # 1 epoch = 1500번 가중치 업데이트 (48000 ÷ 배치크기32)
)

In [ ]:
# 단층 신경망
# - ANN(Artificial Neural Network): 인공 신경망의 총칭
# - 구조: 입력층 → 출력층 (은닉층 없음)
# - 파라미터: 7,850개 (784×10 + 10)

# 심층 신경망
# - DNN(Deep Neural Network): 은닉층이 1개 이상인 신경망
# - 구조: 입력층 → 은닉층 → 출력층
# - 은닉층이 중간 표현(특징)을 자동으로 학습함
# - 딥러닝 기술의 기반

In [ ]:
# 데이터 가져오기 (DNN용 - reshape 없이 원본 2D 형태로)
(train_input, train_target), (test_input, test_target) = keras.datasets.fashion_mnist.load_data()
# 위에서 reshape로 이미 1D로 바꿨기 때문에 재로딩
# DNN 모델은 Input(shape=(784,))을 받으므로 reshape가 필요하지만
# 아래 train_test_split 이후 reshape 적용

In [ ]:
from sklearn.model_selection import train_test_split
# sklearn: 머신러닝 도구 모음 라이브러리
# train_test_split: 데이터를 훈련용/검증용으로 나누는 함수

train_scaled = train_input / 255.0
# 픽셀 0~255 → 0~1 정규화 (학습 안정화 목적)

train_scaled = train_scaled.reshape(-1, 28*28)
# 28×28 → 784 한 줄로 펼침 (-1: 장수 자동 계산)

train_scaled, val_scaled, train_target, val_target = train_test_split(
    train_scaled,    # 전처리된 입력 데이터
    train_target,    # 정답 레이블
    test_size=0.2,   # 전체의 20%를 검증용으로 분리 (12,000장)
    random_state=42  # 분할 결과 고정 (42: 관례적으로 많이 쓰는 시드값)
)

In [ ]:
train_scaled.shape, val_scaled.shape
# 훈련: (48000, 784) - 전체 60000의 80%
# 검증: (12000, 784) - 전체 60000의 20%

In [ ]:
input_layer = keras.layers.Input(shape=(784,))  # 입력층: 784칸 벡터

dense1 = keras.layers.Dense(100, activation='sigmoid')
# Dense(100): 뉴런 100개짜리 완전연결층 (은닉층)
# activation='sigmoid': 각 뉴런 출력을 0~1로 압축
# (후에 ReLU로 교체하면 기울기 소실 없어짐)

dense2 = keras.layers.Dense(10, activation='softmax')
# Dense(10): 출력층, 뉴런 10개 (옷 종류 10가지)
# activation='softmax': 10개를 합이 1인 확률로 변환

In [ ]:
# 방법 1: 미리 정의한 층 변수를 리스트에 담기
model = keras.Sequential([input_layer, dense1, dense2])
# input_layer: 입력층
# dense1: 은닉층 (100뉴런, sigmoid)
# dense2: 출력층 (10뉴런, softmax)
# 순서가 곧 데이터가 흐르는 방향

In [ ]:
# 방법 2: Sequential에 직접 층을 정의하며 이름 지정
model = keras.Sequential([
    keras.layers.Input(shape=(784,)),  # 입력층: 784칸 벡터

    keras.layers.Dense(100, activation='sigmoid', name='hidden_layer'),
    # Dense(100): 은닉층 100뉴런, sigmoid 활성화
    # name='hidden_layer': summary()에서 층 이름으로 표시됨

    keras.layers.Dense(10, activation='softmax', name='output_layer')
    # Dense(10): 출력층 10뉴런, softmax로 확률 분포 출력
    # name='output_layer': summary()에서 층 이름으로 표시됨
])

In [ ]:
model.summary()
# summary(): 모델 구조와 파라미터 수를 표로 출력
# Output Shape의 None: 배치 크기 (실행 시 채워짐)
# hidden_layer params: 784×100 + 100(편향) = 78,500
# output_layer params: 100×10  + 10(편향) =  1,010
# Total params: 79,510

In [ ]:
model.compile(
    optimizer='adam',
    # 'adam': Momentum + 적응적 학습률 결합 옵티마이저
    #   - 파라미터마다 보폭을 자동으로 조절
    #   - 기본 학습률: 0.001

    loss='sparse_categorical_crossentropy',
    # 정수 형태(0~9) 정답에 사용하는 다중 분류 손실 함수

    metrics=['accuracy']
    # 학습 중 화면에 표시할 지표: 정확도(0~1)
)

In [ ]:
# 학습: 훈련 데이터로 가중치 반복 업데이트
model.fit(
    train_scaled,    # 훈련 입력 (48000, 784)
    train_target,    # 훈련 정답 (48000,)
    epochs=5,        # 전체 데이터 5회 반복
                     # 1 epoch = 1500번 업데이트 (48000 ÷ 배치크기32)
    batch_size=32,
    validation_data=(val_scaled, val_target)
    # validation_data: 매 epoch 끝마다 검증 데이터로 성능 측정
    # val_scaled: 검증 입력 (12000, 784)
    # val_target: 검증 정답 (12000,)
    # → val_accuracy로 과적합 여부 확인 가능
)

In [ ]:
# 평가: 학습에 사용하지 않은 검증 데이터로 최종 성능 측정
loss, accuracy = model.evaluate(val_scaled, val_target)
# model.evaluate(): [손실, 정확도] 두 값을 반환
# val_scaled: 처음 보는 검증 입력 (12000, 784)
# val_target: 검증 정답 (12000,)
# loss: 손실값 (낮을수록 좋음)
# accuracy: 정확도 (높을수록 좋음)

print(f'val_loss    : {loss:.4f}')      # .4f: 소수점 4자리까지 출력
print(f'val_accuracy: {accuracy:.4f}')  # 목표: 0.87 이상

In [ ]:
# 데이터 가져오기
# 게이터 전처리 및 분할
# 3번 방법: 시퀀셜에 인라인으로 바로 넣는다.
# 최적화
